In [1]:
!pip install wandb gensim sentence-transformers transformers sentencepiece faiss-cpu peft -q
!pip uninstall -y hf_xet -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import os
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
w_key = user_secrets.get_secret("WANDB_Key")
wandb.login(key=w_key)

import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration, pipeline, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType
import time

def load_with_retry(load_fn, *args, retries=5, delay=8, **kwargs):
    for attempt in range(retries):
        try:
            return load_fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"download failed (attempt {attempt+1}/{retries}), retrying in {delay}s...")
            time.sleep(delay)
OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 11.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 20.9 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [2]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

=== null counts ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== answer distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

=== prompt length (words) ===
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt, dtype: float64

=== avg option length (words) ===
  A: 26.1
  B: 26.5
  C: 26.5
  D: 26.0
  E: 26.2


In [3]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

unique vocab in cleaned prompts: 859
sample — original : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
sample — cleaned  : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


In [4]:
from datasets import Dataset, Dataset as MCQDataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
import torch.nn.functional as F

hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)
print(hf_train)

# tfidf using char n-grams, catches subword patterns better than plain word tokens
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# word2vec on cleaned stopword-removed tokens from scratch
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))
w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

# roberta pretrained model used for cosine sim baseline and attention inspection
bert_tokenizer = load_with_retry(AutoTokenizer.from_pretrained, "roberta-base")
bert_model     = load_with_retry(AutoModel.from_pretrained, "roberta-base", output_attentions=True)
bert_model     = bert_model.to(device)
bert_model.eval()
sample_inputs = bert_tokenizer(train['prompt'].iloc[0], return_tensors="pt").to(device)
with torch.no_grad():
    sample_out = bert_model(**sample_inputs)
print(f"\nattention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")
print("this is how roberta learns which words to focus on relative to others")

# =================================================================
# tfidf + nn (and now LR too) - richer features than before
# NOTE: opt_len and opt_norm are structural features, not content features
# earlier diagnostic showed structural-only features score suspiciously high (0.53 MAP@3)
# WITHOUT even reading the prompt - thats a dataset generation artifact, not real understanding
# keeping cos/jaccard as genuine content signals so we can see which features are doing what
# =================================================================
def jaccard(text1, text2):
    s1, s2 = set(str(text1).lower().split()), set(str(text2).lower().split())
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)

def build_features(row):
    pv = tfidf.transform([row['prompt']])
    option_vecs = {o: tfidf.transform([row[o]]) for o in OPTS}
    feats = {}
    for o in OPTS:
        ov = option_vecs[o]
        cos = float(cosine_similarity(pv, ov)[0][0])                          # content
        jac = jaccard(row['prompt'], row[o])                                  # content
        opt_len  = len(str(row[o]).split())                                   # structural (leak risk)
        opt_norm = float(np.linalg.norm(ov.toarray()))                        # structural (leak risk)
        # average similarity to the other 4 options - how distinct is this option from its siblings
        other_sims = [float(cosine_similarity(ov, option_vecs[o2])[0][0]) for o2 in OPTS if o2 != o]
        avg_other_sim = float(np.mean(other_sims))                            # content-ish
        feats[o] = [cos, jac, opt_len, opt_norm, avg_other_sim]
    return feats

X_rows, y_rows, group_ids = [], [], []
print("\nbuilding features for tfidf classifiers...")
for idx, row in train.iterrows():
    feats = build_features(row)
    for o in OPTS:
        X_rows.append(feats[o])
        y_rows.append(1 if o == row['answer'] else 0)
        group_ids.append(idx)

X_tfidf_feats = np.array(X_rows)
y_tfidf_feats = np.array(y_rows)
group_ids_arr = np.array(group_ids)
print(f"feature matrix: {X_tfidf_feats.shape} (cols: cos, jaccard, opt_len, opt_norm, avg_other_sim)")

# 5-fold cross validation for both LR and MLP, out-of-fold so scores are honest not leaked
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_mlp = np.zeros(len(y_tfidf_feats))
oof_lr  = np.zeros(len(y_tfidf_feats))
unique_groups = train.index.values

for tr_idx, val_idx in kf.split(unique_groups):
    tr_groups = set(unique_groups[tr_idx])
    va_groups = set(unique_groups[val_idx])
    tr_mask = np.array([g in tr_groups for g in group_ids_arr])
    va_mask = np.array([g in va_groups for g in group_ids_arr])

    mlp_fold = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42, early_stopping=True)
    mlp_fold.fit(X_tfidf_feats[tr_mask], y_tfidf_feats[tr_mask])
    oof_mlp[va_mask] = mlp_fold.predict_proba(X_tfidf_feats[va_mask])[:, 1]

    lr_fold = LogisticRegression(max_iter=1000, C=1.0)
    lr_fold.fit(X_tfidf_feats[tr_mask], y_tfidf_feats[tr_mask])
    oof_lr[va_mask] = lr_fold.predict_proba(X_tfidf_feats[va_mask])[:, 1]

print("tfidf mlp + lr cross-validated training done")

# fitting final versions on all train data, these are what get used on test.csv
tfidf_mlp_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42, early_stopping=True)
tfidf_mlp_model.fit(X_tfidf_feats, y_tfidf_feats)

tfidf_lr_model = LogisticRegression(max_iter=1000, C=1.0)
tfidf_lr_model.fit(X_tfidf_feats, y_tfidf_feats)
print("final tfidf mlp and lr classifiers fit on full train data")

print(f"\nlr feature coefficients (cos, jaccard, opt_len, opt_norm, avg_other_sim): {tfidf_lr_model.coef_[0]}")
print("large coefficients on opt_len/opt_norm relative to cos/jaccard would confirm the artifact is doing the work")

# data formatting for mcq - concatenating question with all 5 options into single input string
LETTER2ID = {o: i for i, o in enumerate(OPTS)}
ID2LETTER  = {i: o for o, i in LETTER2ID.items()}

def format_mcq(row):
    return (f"{row['prompt']} "
            f"A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']} E) {row['E']}")

train_inputs = [format_mcq(row) for _, row in train.iterrows()]
train_labels = [LETTER2ID[a] for a in train['answer']]
print(f"\nsample formatted input: {train_inputs[0][:150]}...")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

print("\ncell 4 complete - tfidf (lr+mlp), word2vec, roberta embeddings ready")

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'clean_prompt', 'clean_A', 'clean_B', 'clean_C', 'clean_D', 'clean_E'],
    num_rows: 2000
})
tfidf vocab: 23905
w2v vocab: 2883


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


attention layers: 12, heads per layer: 12
this is how roberta learns which words to focus on relative to others

building features for tfidf classifiers...
feature matrix: (10000, 5) (cols: cos, jaccard, opt_len, opt_norm, avg_other_sim)
tfidf mlp + lr cross-validated training done
final tfidf mlp and lr classifiers fit on full train data

lr feature coefficients (cos, jaccard, opt_len, opt_norm, avg_other_sim): [ 0.32488048 -1.6282322   0.01303107 -0.60520346 -0.41556922]
large coefficients on opt_len/opt_norm relative to cos/jaccard would confirm the artifact is doing the work

sample formatted input: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. A) Mart...

cell 4 complete - tfidf (lr+mlp), word2vec, roberta embeddings ready


In [5]:
# lora roberta - dialing regularization back to a middle ground
# r=16/dropout=0.1 memorized (val acc hit 1.0 fast)
# r=8/dropout=0.3/wd=0.05 underfit (val acc only 0.25, basically random)
# this version splits the difference so the model can actually learn without memorizing
cls_tok  = bert_tokenizer
cls_base = load_with_retry(AutoModelForSequenceClassification.from_pretrained, "roberta-base", num_labels=5)

cls_enc = cls_tok(train_inputs, truncation=True, max_length=256, padding=False)
cls_enc['labels'] = train_labels
hf_cls_full = MCQDataset.from_dict(cls_enc)

hf_split     = hf_cls_full.train_test_split(test_size=0.15, seed=42)
hf_cls_train = hf_split['train']
hf_cls_val   = hf_split['test']
print(f"train: {len(hf_cls_train)} | val: {len(hf_cls_val)}")

full_params = sum(p.numel() for p in cls_base.parameters())
print(f"full model params (if full finetuning): {full_params:,}")

lora_cfg = LoraConfig(
    task_type      = TaskType.SEQ_CLS,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["query", "value"],
    lora_dropout   = 0.15,
    bias           = "none"
)

lora_model = get_peft_model(cls_base, lora_cfg).to(device)
lora_model.print_trainable_parameters()

cls_collator = DataCollatorWithPadding(cls_tok)

cls_args = TrainingArguments(
    output_dir                  = "/kaggle/working/lora_roberta",
    num_train_epochs            = 15,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-4,
    weight_decay                = 0.01,
    warmup_steps                = 50,
    fp16                        = True,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    greater_is_better           = True,
    report_to                   = "none",
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
    seed                        = 42
)

cls_trainer = Trainer(
    model            = lora_model,
    args             = cls_args,
    train_dataset    = hf_cls_train,
    eval_dataset     = hf_cls_val,
    data_collator    = cls_collator,
    processing_class = cls_tok,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("starting lora roberta finetuning (r=16, dropout=0.15, up to 15 epochs, early stopping)...")
cls_trainer.train()
lora_model.eval()
print("lora roberta finetuning done — using best checkpoint by validation accuracy")

roberta_val = cls_trainer.evaluate()
print(f"\nroberta final validation accuracy: {roberta_val['eval_accuracy']:.4f}")

print("\nsanity check on first 5 train rows:")
for i in range(5):
    row_i = train.iloc[i]
    ins_i = cls_tok(format_mcq(row_i), return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits_i = lora_model(**ins_i).logits[0]
    pred_i = ID2LETTER[int(torch.argmax(logits_i))]
    print(f"row {i} | predicted: '{pred_i}' | actual: {row_i['answer']}")

print(f"\ncell 5 complete - lora roberta trained, val accuracy: {roberta_val['eval_accuracy']:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

train: 1700 | val: 300
full model params (if full finetuning): 124,649,477
trainable params: 1,184,261 || all params: 125,833,738 || trainable%: 0.9411
starting lora roberta finetuning (r=16, dropout=0.15, up to 15 epochs, early stopping)...
{'eval_loss': '3.216', 'eval_accuracy': '0.24', 'eval_runtime': '2.397', 'eval_samples_per_second': '125.1', 'eval_steps_per_second': '2.086', 'epoch': '1'}
{'loss': '6.412', 'grad_norm': '4.492', 'learning_rate': '9.8e-05', 'epoch': '1.852'}
{'eval_loss': '3.193', 'eval_accuracy': '0.2367', 'eval_runtime': '2.6', 'eval_samples_per_second': '115.4', 'eval_steps_per_second': '1.923', 'epoch': '2'}
{'eval_loss': '3.163', 'eval_accuracy': '0.3033', 'eval_runtime': '3.028', 'eval_samples_per_second': '99.06', 'eval_steps_per_second': '1.651', 'epoch': '3'}
{'loss': '6.251', 'grad_norm': '5.429', 'learning_rate': '8.62e-05', 'epoch': '3.704'}
{'eval_loss': '2.739', 'eval_accuracy': '0.5', 'eval_runtime': '2.727', 'eval_samples_per_second': '110', 'eval_

In [6]:
tok_bert  = load_with_retry(AutoTokenizer.from_pretrained, "bert-base-uncased")
base_bert = load_with_retry(AutoModelForSequenceClassification.from_pretrained, "bert-base-uncased", num_labels=5)

enc_bert = tok_bert(train_inputs, truncation=True, max_length=256, padding=False)
enc_bert['labels'] = train_labels
hf_bert_full = MCQDataset.from_dict(enc_bert)

hf_bert_split = hf_bert_full.train_test_split(test_size=0.15, seed=42)
hf_bert_train = hf_bert_split['train']
hf_bert_val   = hf_bert_split['test']
print(f"train: {len(hf_bert_train)} | val: {len(hf_bert_val)}")

lora_cfg_bert = LoraConfig(
    task_type      = TaskType.SEQ_CLS,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["query", "value"],
    lora_dropout   = 0.15,
    bias           = "none"
)

lora_model_bert = get_peft_model(base_bert, lora_cfg_bert).to(device)
lora_model_bert.print_trainable_parameters()

collator_bert = DataCollatorWithPadding(tok_bert)

args_bert = TrainingArguments(
    output_dir                  = "/kaggle/working/lora_bert",
    num_train_epochs            = 15,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-4,
    weight_decay                = 0.01,
    warmup_steps                = 50,
    fp16                        = True,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    greater_is_better           = True,
    report_to                   = "none",
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
    seed                        = 42
)

trainer_bert = Trainer(
    model            = lora_model_bert,
    args             = args_bert,
    train_dataset    = hf_bert_train,
    eval_dataset     = hf_bert_val,
    data_collator    = collator_bert,
    processing_class = tok_bert,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("starting lora bert finetuning (r=16, dropout=0.15, up to 15 epochs, early stopping)...")
trainer_bert.train()
lora_model_bert.eval()
print("lora bert finetuning done — using best checkpoint by validation accuracy")

bert_val = trainer_bert.evaluate()
print(f"\nbert final validation accuracy: {bert_val['eval_accuracy']:.4f}")

print("\nsanity check on first 5 train rows:")
for i in range(5):
    row_i = train.iloc[i]
    ins_i = tok_bert(format_mcq(row_i), return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits_i = lora_model_bert(**ins_i).logits[0]
    pred_i = ID2LETTER[int(torch.argmax(logits_i))]
    print(f"row {i} | predicted: '{pred_i}' | actual: {row_i['answer']}")

print(f"\ncell 6 complete - lora bert trained, val accuracy: {bert_val['eval_accuracy']:.4f}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

train: 1700 | val: 300
trainable params: 593,669 || all params: 110,079,754 || trainable%: 0.5393
starting lora bert finetuning (r=16, dropout=0.15, up to 15 epochs, early stopping)...
{'eval_loss': '3.184', 'eval_accuracy': '0.2467', 'eval_runtime': '3.184', 'eval_samples_per_second': '94.22', 'eval_steps_per_second': '1.57', 'epoch': '1'}
{'loss': '6.434', 'grad_norm': '10.28', 'learning_rate': '9.8e-05', 'epoch': '1.852'}
{'eval_loss': '3.12', 'eval_accuracy': '0.31', 'eval_runtime': '2.914', 'eval_samples_per_second': '103', 'eval_steps_per_second': '1.716', 'epoch': '2'}
{'eval_loss': '2.956', 'eval_accuracy': '0.34', 'eval_runtime': '3.083', 'eval_samples_per_second': '97.31', 'eval_steps_per_second': '1.622', 'epoch': '3'}
{'loss': '6.007', 'grad_norm': '7.545', 'learning_rate': '8.62e-05', 'epoch': '3.704'}
{'eval_loss': '2.722', 'eval_accuracy': '0.39', 'eval_runtime': '2.981', 'eval_samples_per_second': '100.6', 'eval_steps_per_second': '1.677', 'epoch': '4'}
{'eval_loss': '2

In [7]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

def get_roberta_embedding(text):
    inputs     = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():
        out    = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask       = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    summed     = torch.sum(token_embs * mask, dim=1)
    counts     = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / counts).squeeze(0).cpu().numpy()

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_tfidf_mlp(row):
    feats = build_features(row)
    scores = {o: tfidf_mlp_model.predict_proba([feats[o]])[0][1] for o in OPTS}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def rank_tfidf_lr(row):
    feats = build_features(row)
    scores = {o: tfidf_lr_model.predict_proba([feats[o]])[0][1] for o in OPTS}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_roberta(row):
    pv   = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def get_member_probs(model, tok, text):
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    return F.softmax(logits, dim=-1).cpu().numpy()

def rank_ensemble_weighted(row):
    text        = format_mcq(row)
    probs_rob   = get_member_probs(lora_model, cls_tok, text)
    probs_bert  = get_member_probs(lora_model_bert, tok_bert, text)
    total_acc   = roberta_val['eval_accuracy'] + bert_val['eval_accuracy']
    w_rob       = roberta_val['eval_accuracy'] / total_acc
    w_bert      = bert_val['eval_accuracy'] / total_acc
    final_probs = w_rob * probs_rob + w_bert * probs_bert
    scores      = {ID2LETTER[i]: final_probs[i] for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def rank_fusion(row, w_rob=0.6, w_bert=0.4):
    text       = format_mcq(row)
    probs_rob  = get_member_probs(lora_model, cls_tok, text)
    probs_bert = get_member_probs(lora_model_bert, tok_bert, text)
    rank_rob   = {ID2LETTER[i]: r for r, i in enumerate(np.argsort(-probs_rob))}
    rank_bert  = {ID2LETTER[i]: r for r, i in enumerate(np.argsort(-probs_bert))}
    fused = {o: w_rob * rank_rob[o] + w_bert * rank_bert[o] for o in OPTS}
    return sorted(fused, key=lambda x: fused[x])

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

from sklearn.metrics import accuracy_score, f1_score

def top1_accuracy(actuals, predictions):
    return round(accuracy_score(actuals, [p[0] for p in predictions]), 4)

def top1_f1(actuals, predictions):
    return round(f1_score(actuals, [p[0] for p in predictions], average='macro'), 4)

actuals     = train['answer'].tolist()
top3_maj    = list(train['answer'].value_counts().index[:3])
map3_maj    = round(mapk(actuals, [top3_maj] * len(train)), 4)

tfidf_preds = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf  = round(mapk(actuals, tfidf_preds), 4)
acc_tfidf   = top1_accuracy(actuals, tfidf_preds)
f1_tfidf    = top1_f1(actuals, tfidf_preds)

# tfidf mlp/lr scored via out-of-fold predictions from cell 4, honest and not leaked
tfidf_mlp_preds, tfidf_lr_preds = [], []
oof_idx = 0
for _, row in train.iterrows():
    scores_mlp = {o: oof_mlp[oof_idx + i] for i, o in enumerate(OPTS)}
    scores_lr  = {o: oof_lr[oof_idx + i]  for i, o in enumerate(OPTS)}
    oof_idx += 5
    tfidf_mlp_preds.append(sorted(scores_mlp, key=lambda x: scores_mlp[x], reverse=True)[:3])
    tfidf_lr_preds.append(sorted(scores_lr, key=lambda x: scores_lr[x], reverse=True)[:3])

map3_tfidf_mlp = round(mapk(actuals, tfidf_mlp_preds), 4)
acc_tfidf_mlp  = top1_accuracy(actuals, tfidf_mlp_preds)
f1_tfidf_mlp   = top1_f1(actuals, tfidf_mlp_preds)

map3_tfidf_lr  = round(mapk(actuals, tfidf_lr_preds), 4)
acc_tfidf_lr   = top1_accuracy(actuals, tfidf_lr_preds)
f1_tfidf_lr    = top1_f1(actuals, tfidf_lr_preds)

w2v_preds   = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v    = round(mapk(actuals, w2v_preds), 4)
acc_w2v     = top1_accuracy(actuals, w2v_preds)
f1_w2v      = top1_f1(actuals, w2v_preds)

sample_roberta  = train.sample(n=200, random_state=42)
roberta_preds   = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
map3_roberta    = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)
acc_roberta_cos = top1_accuracy(sample_roberta['answer'].tolist(), roberta_preds)
f1_roberta_cos  = top1_f1(sample_roberta['answer'].tolist(), roberta_preds)

print("evaluating weighted ensemble on full train (reference, will look inflated)...")
ens_preds = [rank_ensemble_weighted(row)[:3] for _, row in train.iterrows()]
map3_ens  = round(mapk(actuals, ens_preds), 4)
acc_ens   = top1_accuracy(actuals, ens_preds)
f1_ens    = top1_f1(actuals, ens_preds)

print("evaluating rank fusion on full train (reference, will look inflated)...")
rankfusion_preds = [rank_fusion(row)[:3] for _, row in train.iterrows()]
map3_rankfusion   = round(mapk(actuals, rankfusion_preds), 4)
acc_rankfusion    = top1_accuracy(actuals, rankfusion_preds)
f1_rankfusion     = top1_f1(actuals, rankfusion_preds)

print(f"\nmajority baseline          MAP@3 : {map3_maj}")
print(f"tfidf cosine               MAP@3 : {map3_tfidf}")
print(f"tfidf + mlp (OOF, honest)  MAP@3 : {map3_tfidf_mlp}")
print(f"tfidf + lr (OOF, honest)   MAP@3 : {map3_tfidf_lr}")
print(f"word2vec cosine            MAP@3 : {map3_w2v}")
print(f"roberta cosine (200)       MAP@3 : {map3_roberta}")
print(f"roberta VAL accuracy (honest) : {roberta_val['eval_accuracy']:.4f}")
print(f"bert VAL accuracy (honest)    : {bert_val['eval_accuracy']:.4f}")
print(f"weighted ensemble (full train, inflated) MAP@3 : {map3_ens}")
print(f"rank fusion (full train, inflated) MAP@3 : {map3_rankfusion}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone5-full-ensemble-v2",
    config  = {
        "milestone": 5, "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.15,
        "weight_decay": 0.01, "lr": 1e-4, "max_epochs": 15,
        "ensemble": ["roberta-base", "bert-base-uncased"],
        "tfidf_features": "cos, jaccard, opt_len, opt_norm, avg_other_sim"
    },
    tags = ["milestone-5", "lora", "ensemble", "tfidf-lr-mlp"]
)
wandb.log({
    "map3/majority_baseline"     : map3_maj,
    "map3/tfidf_cosine"          : map3_tfidf,
    "map3/tfidf_mlp_oof"         : map3_tfidf_mlp,
    "map3/tfidf_lr_oof"          : map3_tfidf_lr,
    "map3/word2vec_cosine"       : map3_w2v,
    "map3/roberta_cosine_200"    : map3_roberta,
    "accuracy/roberta_val_honest": roberta_val['eval_accuracy'],
    "accuracy/bert_val_honest"   : bert_val['eval_accuracy'],
    "map3/ensemble_fulltrain_ref"   : map3_ens,
    "map3/rankfusion_fulltrain_ref" : map3_rankfusion
})
wandb.finish()

best_technique = "rank_fusion" if map3_rankfusion > map3_ens else "weighted"
print(f"\nusing {best_technique} ensemble for submission")

test_preds = []
for _, row in test.iterrows():
    ranked = rank_fusion(row) if best_technique == "rank_fusion" else rank_ensemble_weighted(row)
    test_preds.append(" ".join(ranked[:3]))

submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
submission.head(5)

evaluating weighted ensemble on full train (reference, will look inflated)...
evaluating rank fusion on full train (reference, will look inflated)...

majority baseline          MAP@3 : 0.4213
tfidf cosine               MAP@3 : 0.3105
tfidf + mlp (OOF, honest)  MAP@3 : 0.2971
tfidf + lr (OOF, honest)   MAP@3 : 0.4904
word2vec cosine            MAP@3 : 0.3094
roberta cosine (200)       MAP@3 : 0.3425
roberta VAL accuracy (honest) : 0.9633
bert VAL accuracy (honest)    : 0.8067
weighted ensemble (full train, inflated) MAP@3 : 0.9916
rank fusion (full train, inflated) MAP@3 : 0.993

using rank_fusion ensemble for submission
submission.csv saved


,ID,Prediction
0,1,A C E
1,2,B C A
2,3,B C A
3,4,E C D
4,5,C E D
